# The Chain Rule & Backpropagation

## What's covered

- The **chain rule** for one variable, written so it scales
- The **multivariable chain rule** — for compositions of vector-valued functions
- **Computational graphs** — the data structure backprop sees
- **Forward mode** vs **reverse mode** automatic differentiation — and why reverse mode wins for neural networks
- **Backpropagation derived from scratch** on a tiny 2-layer MLP
- Where this appears in ML — every framework, every training step


## The chain rule, again, written so it scales

We met it briefly in notebook 2. Here we write it the way it actually gets used.

**Single-variable form.** If `y = f(g(x))`, and we set `u = g(x)`, then

$$
\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}
$$

The `du` "cancels" — a mnemonic, not a proof, but it captures the truth: derivatives **multiply** through compositions.

**Why this scales.** Suppose `y = f_4(f_3(f_2(f_1(x))))`. Chain the rule again and again:

$$
\frac{dy}{dx} = \frac{dy}{du_3} \cdot \frac{du_3}{du_2} \cdot \frac{du_2}{du_1} \cdot \frac{du_1}{dx}
$$

Each layer contributes one factor. A neural network is exactly this — many layers stacked, each one a function applied to the previous one's output. Computing `dy/dx` means **multiplying a sequence of layer-local derivatives.**

That's it. There is nothing else to backpropagation conceptually. The rest is bookkeeping: in what order do you multiply, and how do you avoid recomputing the same thing twice. Spoiler: the answer is *right to left*, and that ordering is what gives us *reverse-mode* automatic differentiation.


In [ ]:
import numpy as np

# Verify the chain rule on y = sin(x^2 + 1)
# Setting u = x^2 + 1, dy/du = cos(u), du/dx = 2x → dy/dx = 2x cos(x^2 + 1)
f = lambda x: np.sin(x ** 2 + 1)
df_analytic = lambda x: 2 * x * np.cos(x ** 2 + 1)

def central_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print(f"{'x':>6}  {'analytic dy/dx':>16}  {'numerical':>14}")
for x in [-1.5, -0.5, 0.0, 0.5, 1.5]:
    print(f"{x:>6}  {df_analytic(x):>16.6f}  {central_diff(f, x):>14.6f}")


## The multivariable chain rule

A neural network's intermediate variables aren't scalars — they are vectors (layer activations) and even matrices (batched activations). The chain rule generalizes naturally.

**Setup.** Let `\mathbf{u} = g(\mathbf{x})` be a vector-valued function of a vector, and `y = f(\mathbf{u})` be a scalar-valued function of that vector. Then

$$
\frac{\partial y}{\partial x_i} = \sum_j \frac{\partial y}{\partial u_j} \cdot \frac{\partial u_j}{\partial x_i}
$$

Each path from `x_i` through some `u_j` to `y` contributes a product of partials; sum over all paths. This is the chain rule restated for multivariable composition.

**Compactly, in matrix form.** Stacking everything as vectors and matrices,

$$
\nabla_{\mathbf{x}} y = J_g^T \, \nabla_{\mathbf{u}} y
$$

where `J_g` is the **Jacobian** of `g` (its `(j, i)` entry is `∂u_j/∂x_i`). We'll meet Jacobians formally in the next notebook; for now think of `J_g^T \nabla y` as the matrix-vector version of "multiply by the layer-local derivative."

**The picture for a deep network.** Each layer of a network is a function `u_{k+1} = f_k(u_k, θ_k)`. The gradient of the final loss `L` with respect to any intermediate `u_k` is obtained by **repeatedly multiplying by Jacobians, walking backward from the loss**:

$$
\nabla_{\mathbf{u}_k} L = J_{f_k}^T \, J_{f_{k+1}}^T \, \dots \, J_{f_{n-1}}^T \, \nabla_{\mathbf{u}_n} L
$$

That is, *literally*, backpropagation. The rest of this notebook makes it concrete on a tiny example.


## Computational graphs

A **computational graph** is a directed acyclic graph in which each node is an elementary operation and each edge carries a value (the operation's output). Inputs flow in at the leaves; the final scalar (loss) is at the root.

For example, `z = (x + y) * (y + 2)` becomes:

```
    x ──┐
        ├─► (+) ──► a ──┐
    y ──┤               ├─► (*) ──► z
        │   2 ──► (+) ──► b ──┘
        └───────────┘
```

Two of `y`'s contributions reach `z` via two different paths (`a` and `b`).

**Forward pass.** Walk leaves → root, computing each node's value. Standard evaluation.

**Backward pass (reverse-mode autodiff / backpropagation).** Starting at the root with `∂z/∂z = 1`, walk root → leaves. At each node, multiply the incoming gradient by the local derivative, and split it across the node's inputs. Sum contributions where paths recombine.

**Why reverse mode?** A neural network computes one scalar loss from many parameters (millions to billions). Reverse mode computes the gradient of *one* output with respect to *all* inputs in a single backward pass — same cost as one forward pass. Forward mode would do the opposite: one input vs all outputs in one sweep. Wrong asymmetry for ML, where we have far more parameters than losses.

That asymmetry is *the* engineering insight. Everything that distinguishes a modern ML framework (PyTorch, JAX, TensorFlow) is built around running reverse-mode autodiff efficiently — caching intermediate values, reusing them, parallelizing across devices.


In [ ]:
# Backprop by hand on z = (x + y) * (y + 2), at x = 3, y = 4
x, y = 3.0, 4.0

# Forward pass — store every intermediate
a = x + y          # 7
b = y + 2          # 6
z = a * b          # 42
print(f"forward: x={x}, y={y}, a={a}, b={b}, z={z}")

# Backward pass — gradients walk back through the graph
dz = 1.0           # seed: ∂z/∂z = 1
da = dz * b        # ∂z/∂a = b
db = dz * a        # ∂z/∂b = a

dx = da * 1                 # x contributes only through a
dy_from_a = da * 1          # y contributes through a
dy_from_b = db * 1          # y also contributes through b
dy = dy_from_a + dy_from_b  # SUM contributions where paths recombine

print(f"backward: ∂z/∂x = {dx},  ∂z/∂y = {dy}")

# Verify ∂z/∂x and ∂z/∂y numerically
def Z(x, y): return (x + y) * (y + 2)
print(f"numerical ∂z/∂x = {(Z(x + 1e-6, y) - Z(x - 1e-6, y)) / 2e-6:.4f}")
print(f"numerical ∂z/∂y = {(Z(x, y + 1e-6) - Z(x, y - 1e-6)) / 2e-6:.4f}")


## Backprop on a tiny MLP — the math

Now the calculation that powers every training run. We pick the smallest non-trivial neural network: **one hidden layer, sigmoid activation, scalar output, MSE loss.**

**Architecture.**

$$
\mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \mathbf{w}_2^T \mathbf{h} + b_2
$$

with input `x ∈ R^d`, hidden width `m`, target `y`, loss `L = (1/2)(\hat{y} - y)^2`.

**Goal.** Compute `∂L/∂W_1`, `∂L/∂b_1`, `∂L/∂w_2`, `∂L/∂b_2`.

**Forward pass** (saves activations for backward).

```
z₁ = W₁ x + b₁          (pre-activation, vector of length m)
h  = σ(z₁)              (hidden activation, length m)
ŷ  = w₂ᵀ h + b₂          (scalar)
L  = ½ (ŷ - y)²         (scalar)
```

**Backward pass.** Apply the chain rule, walking from `L` back to inputs.

$$
\frac{\partial L}{\partial \hat{y}} = \hat{y} - y \qquad \text{(call this } \delta_{\text{out}} \text{)}
$$

Output layer:

$$
\frac{\partial L}{\partial \mathbf{w}_2} = \delta_{\text{out}} \cdot \mathbf{h}, \qquad \frac{\partial L}{\partial b_2} = \delta_{\text{out}}
$$

Push the signal back into the hidden layer:

$$
\frac{\partial L}{\partial \mathbf{h}} = \delta_{\text{out}} \cdot \mathbf{w}_2
$$

Through the sigmoid (using `σ'(z) = σ(z)(1 - σ(z)) = h(1 - h)`):

$$
\frac{\partial L}{\partial \mathbf{z}_1} = \frac{\partial L}{\partial \mathbf{h}} \odot \mathbf{h} \odot (1 - \mathbf{h}) \qquad \text{(call this } \boldsymbol{\delta}_1 \text{)}
$$

(`⊙` is element-wise product.) And finally:

$$
\frac{\partial L}{\partial W_1} = \boldsymbol{\delta}_1 \, \mathbf{x}^T, \qquad \frac{\partial L}{\partial \mathbf{b}_1} = \boldsymbol{\delta}_1
$$

That's it. Every neural network you train is one variant of this calculation, repeated for every layer.


In [ ]:
# Tiny MLP from scratch — forward + backward + gradient check
rng = np.random.default_rng(0)
d, m = 4, 3   # input dim, hidden width

W1 = rng.normal(size=(m, d))
b1 = rng.normal(size=m)
w2 = rng.normal(size=m)
b2 = rng.normal()

x = rng.normal(size=d)
y = 0.5

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def forward(W1, b1, w2, b2, x):
    z1 = W1 @ x + b1
    h  = sigmoid(z1)
    y_hat = w2 @ h + b2
    L = 0.5 * (y_hat - y) ** 2
    return L, (z1, h, y_hat)

L, (z1, h, y_hat) = forward(W1, b1, w2, b2, x)
print(f"forward loss = {L:.6f},  ŷ = {y_hat:.4f}, y = {y}")

# Backward pass — apply chain rule layer by layer
delta_out = y_hat - y                         # ∂L/∂ŷ
dw2 = delta_out * h                           # ∂L/∂w₂
db2 = delta_out                               # ∂L/∂b₂
dh = delta_out * w2                           # ∂L/∂h
delta1 = dh * h * (1 - h)                     # ∂L/∂z₁ via σ'(z) = h(1-h)
dW1 = np.outer(delta1, x)                     # ∂L/∂W₁
db1 = delta1                                  # ∂L/∂b₁

# Gradient check via finite differences on b1
def L_fn(b1_perturbed):
    return forward(W1, b1_perturbed, w2, b2, x)[0]

db1_num = np.zeros_like(b1)
for i in range(m):
    bp, bm = b1.copy(), b1.copy()
    bp[i] += 1e-5
    bm[i] -= 1e-5
    db1_num[i] = (L_fn(bp) - L_fn(bm)) / 2e-5

print("\nbackprop ∂L/∂b1 :", db1.round(6))
print("numerical ∂L/∂b1:", db1_num.round(6))
print("max abs diff   :", np.max(np.abs(db1 - db1_num)))


## Forward vs reverse mode — the asymmetry that matters

Forward and reverse mode autodiff both compute *exact* derivatives by applying the chain rule. They differ in **direction of accumulation**, and the choice has dramatic consequences for cost.

**Forward mode.** Carry a derivative *forward* with each variable. At each node, push the chain rule forward.

- **Output:** `∂y/∂x` for *one* fixed input `x` versus *all* outputs `y`.
- **Cost:** `O(n_inputs)` forward sweeps — one per input.
- **Best when:** you have few inputs and many outputs.

**Reverse mode (backpropagation).** Do the forward pass storing intermediates, then run one backward sweep accumulating gradients.

- **Output:** `∂L/∂θ` for *one* fixed scalar output `L` versus *all* inputs `θ`.
- **Cost:** `O(1)` backward sweeps — one for the loss.
- **Best when:** you have many inputs (millions of weights) and one output (the loss).

ML lives squarely in reverse-mode territory. The price for reverse mode is **memory** — you must store every intermediate activation during the forward pass so the backward pass can use them. That's why training deep networks needs so much GPU RAM, and why techniques like gradient checkpointing (trade compute for memory by recomputing activations on demand) exist.

**Jacobian-vector products (JVP) and vector-Jacobian products (VJP).** In modern parlance:

- Forward mode = JVP (`v → J v`)
- Reverse mode = VJP (`v → v^T J`)

These are the primitives PyTorch and JAX expose. `loss.backward()` is *literally* a chain of VJPs.


## Where this appears in ML

Backpropagation is the single most consequential algorithm in modern ML. Every training run is one application of it.

- **Every neural network training step.** Forward → loss → backward (chain rule) → parameter update. That's the loop.
- **PyTorch, JAX, TensorFlow.** Their `autograd` engines build computational graphs (eagerly in PyTorch, traced in JAX) and run reverse-mode autodiff. `loss.backward()` traverses the graph applying the chain rule.
- **Gradient checkpointing.** Trade compute for memory by recomputing activations on demand during the backward pass instead of storing them all. Crucial for training transformers with long context.
- **Mixed-precision training.** Activations stored in `float16` to save memory; gradients computed and accumulated in `float32` to preserve accuracy. The chain rule has to be *carefully* implemented across precisions.
- **Differentiable programming.** Any program written with differentiable primitives can be backpropagated through — differentiable physics simulators, differentiable rasterizers, differentiable database queries. All chain rule on bigger graphs.
- **Higher-order gradients.** Want `∂²L/∂θ²` (the Hessian)? Run backprop twice — once to get `∂L/∂θ`, once more on `∂L/∂θ` as a function. JAX's `jax.grad(jax.grad(f))` does this.
- **Implicit differentiation.** When the forward pass involves solving an equation (`OptNet`, MAML, deep equilibrium models), the implicit function theorem extends backprop through the solver.
- **Reverse-mode autodiff isn't just for neural nets.** Bayesian optimization uses it for hyperparameter gradients; differentiable rendering uses it for scene reconstruction; even SQL query optimizers have been trained with it.

Next notebook: **Jacobians, Hessians, and matrix calculus** — we formalize the Jacobian and Hessian objects we've been hand-waving at, and build the matrix-calculus identity table you'll use on every whiteboard.
